# Fit Maxima and Minima
- This uses the labels.json to for the maxima and minima to extract the uncertainties for the predictions thereof in our analysis.
- For the ones that had bad data or seemed constant under the ubservation noise, we can only report MSE or loss or whatever.
---
- We can use the fitted means of the parabola to be the times of the maxima and the minima.
- Use the curve fit uncertainties as the uncertainties on those times.

In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.optimize import curve_fit

import celerite2
from celerite2 import terms

from prettytable import PrettyTable

import json

import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

from helpers.df_ops import prepare_df, split_df, clean_df
from helpers.priors import find_classify_signals, get_priors


In [73]:
# Loop through the data files and the JSON. 
# For ones that were not marked as bad or constant, fit a parabola to the peak.
def maxima(x, mu, C, A):
    return C - A*((x-mu)**2)

def minima(x, mu, C, A):
    return C + A*((x-mu)**2)

def fit_parabola(
    data_df,
    curve_type: str = None, # "max" or "min"
    mu_pos: float = None, # the postion labelled manually in label tool and saved in JSON
    mu_bound = 365 * 1,
    window_mult = 1.5 # how many times mu_bound
    ):
    '''
    Returns (mu, mu_std), (C, C_std), (A, A_std)
    '''
    window = window_mult * mu_bound
    window_df = data_df[
        (data_df['day'] > (mu_pos - window)) &
        (data_df['day'] < (mu_pos + window))
    ]

    C_guess = float(np.interp(mu_pos, data_df['day'].values, data_df['sind'].values))
    A_guess = 1
    initial_guess = [mu_pos, C_guess, A_guess]

    mu_bounds = (mu_pos - mu_bound, mu_pos + mu_bound)
    C_bounds  = (0, np.inf)
    A_bounds  = (0, np.inf)

    # Clip initial guesses to lie within bounds (avoids ValueError from curve_fit)
    initial_guess = [
        float(np.clip(mu_pos,  mu_bounds[0], mu_bounds[1])),
        float(np.clip(C_guess, C_bounds[0],  1e15)),
        float(np.clip(A_guess, A_bounds[0],  1e15)),
    ]

    mm_func = maxima if curve_type == "max" else minima
    popt, pcov = curve_fit(
        mm_func,
        window_df['day'],
        window_df['sind'],
        p0=initial_guess,
        bounds=(
            [mu_bounds[0], C_bounds[0], A_bounds[0]],
            [mu_bounds[1], C_bounds[1], A_bounds[1]],
        )
    )

    mu_ret = (popt[0], np.sqrt(pcov[0][0]))
    C_ret  = (popt[1], np.sqrt(pcov[1][1]))
    A_ret  = (popt[2], np.sqrt(pcov[2][2]))

    return mu_ret, C_ret, A_ret

In [74]:
DATA_DIR   = Path('../Data')
LABELS_PATH = Path('labels.json')

with open(LABELS_PATH) as f:
    labels = json.load(f)

parabola_results = {}

for fname, entry in labels.items():
    if entry.get('bad') or entry.get('const'): # If either of these is true
        continue # We do not try fitting a parabola to it

    fpath = next(DATA_DIR.rglob(fname), None) # See if we can find the concomitant file
    if fpath is None:
        print(f"File not found: {fname}")
        continue

    raw_df = pd.read_csv(fpath, sep=r'\s+', skip_blank_lines=True)

    # Prepare the df in relative dates
    try:
        data_df = prepare_df(raw_df, add_prefix=False, relative=True)
    except Exception:
        data_df = prepare_df(raw_df, add_prefix=True, relative=True) # Relative  = True here only shifts the internal JD the days/years still absolute

    # shift day to match what the labeller saw
    data_df = data_df.copy()
    data_df['day'] -= data_df['day'].min() # Relative dates 


    # Add the maxima and minima
    file_results = {'maxima': [], 'minima': []}

    # Fit the parabola
    for mu_pos in entry.get('maxima', []):
        mu, C, A = fit_parabola(data_df, curve_type='max', mu_pos=mu_pos)
        file_results['maxima'].append({'mu': mu, 'C': C, 'A': A})

    for mu_pos in entry.get('minima', []):
        mu, C, A = fit_parabola(data_df, curve_type='min', mu_pos=mu_pos)
        file_results['minima'].append({'mu': mu, 'C': C, 'A': A})

    parabola_results[fname] = file_results

In [75]:
TOL = 6

def load_clean(fname):
    fpath = next(DATA_DIR.rglob(fname), None)
    if fpath is None:
        raise FileNotFoundError(fname)
    raw_df = pd.read_csv(fpath, sep=r'\s+', skip_blank_lines=True)
    try:
        df = prepare_df(raw_df, add_prefix=False, relative=True)
    except Exception:
        df = prepare_df(raw_df, add_prefix=True, relative=True)
    df = df.copy()
    df['day'] -= df['day'].min()
    med = df['sind'].median()
    mad = (df['sind'] - med).abs().median()
    return (
        df[(df['sind'] - med).abs() < TOL * mad]
        .sort_values('day')
        .reset_index(drop=True)
    )

def gaussian_smooth(t, y, sigma, n_eval=500):
    t_eval = np.linspace(t.min(), t.max(), n_eval)
    w = np.exp(-0.5 * ((t[:, None] - t_eval[None, :]) / sigma) ** 2)
    return t_eval, (w * y[:, None]).sum(axis=0) / w.sum(axis=0)

fig, axes = plt.subplots(len(labels), 1, figsize=(20, 5 * len(labels)))
if len(labels) == 1:
    axes = [axes]

for ax, (fname, lbl) in zip(axes, labels.items()):
    try:
        data_df = load_clean(fname)
    except Exception as e:
        ax.text(0.5, 0.5, f"Could not load:\n{e}", transform=ax.transAxes,
                ha='center', va='center', color='red', fontsize=10)
        ax.set_title(fname)
        continue

    is_const = lbl.get('const', False)
    is_bad   = lbl.get('bad',   False)

    color = 'steelblue' if not is_bad else 'salmon'
    ax.plot(data_df['day'], data_df['sind'], '.', color=color,
            markersize=3, alpha=0.7, rasterized=True)

    span = data_df['day'].max() - data_df['day'].min()
    t_s, y_s = gaussian_smooth(data_df['day'].values, data_df['sind'].values, sigma=span * 0.04)
    ax.plot(t_s, y_s, '-', color='black', linewidth=1.5, alpha=0.6, zorder=3)

    if is_bad:
        ax.set_facecolor('#fff0f0')
        ax.text(0.5, 0.5, 'BAD DATA', transform=ax.transAxes,
                fontsize=28, color='red', alpha=0.3,
                ha='center', va='center', fontweight='bold')
    elif is_const:
        ax.set_facecolor('#fff8e1')
        ax.text(0.5, 0.5, 'CONST', transform=ax.transAxes,
                fontsize=28, color='#ffb300', alpha=0.4,
                ha='center', va='center', fontweight='bold')
    else:
        y_max = data_df['sind'].max()
        y_min = data_df['sind'].min()
        y_pad = (y_max - y_min) * 0.03

        for day in lbl['maxima']:
            ax.axvline(day, color='royalblue', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.text(day, y_max + y_pad, 'MAX', color='royalblue',
                    fontsize=7, ha='center', va='bottom', rotation=90)

        for day in lbl['minima']:
            ax.axvline(day, color='darkorange', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.text(day, y_min - y_pad, 'MIN', color='darkorange',
                    fontsize=7, ha='center', va='top', rotation=90)

        # Overlay fitted parabolas; shaded vertical band shows mu +/- mu_std
        if fname in parabola_results:
            fit_window = 365 * 1.5
            for fit in parabola_results[fname]['maxima']:
                mu_val, mu_std = fit['mu']
                C_val = fit['C'][0]
                A_val = fit['A'][0]
                t_fit = np.linspace(mu_val - fit_window, mu_val + fit_window, 300)
                ax.plot(t_fit, maxima(t_fit, mu_val, C_val, A_val),
                        '-', color='royalblue', linewidth=2, alpha=0.85, zorder=4)
                ax.axvline(mu_val, color='royalblue', linewidth=1.5, zorder=4)
                ax.axvspan(mu_val - mu_std, mu_val + mu_std, color='royalblue', alpha=0.15, zorder=3)

            for fit in parabola_results[fname]['minima']:
                mu_val, mu_std = fit['mu']
                C_val = fit['C'][0]
                A_val = fit['A'][0]
                t_fit = np.linspace(mu_val - fit_window, mu_val + fit_window, 300)
                ax.plot(t_fit, minima(t_fit, mu_val, C_val, A_val),
                        '-', color='darkorange', linewidth=2, alpha=0.85, zorder=4)
                ax.axvline(mu_val, color='darkorange', linewidth=1.5, zorder=4)
                ax.axvspan(mu_val - mu_std, mu_val + mu_std, color='darkorange', alpha=0.15, zorder=3)

    tags = ''.join(f'  [{t.upper()}]' for t in ('const', 'bad') if lbl.get(t, False))
    ax.set_title(f"{fname}{tags}")
    ax.set_xlabel("Days since first observation")
    ax.set_ylabel("S-index")

plt.tight_layout()
plt.show()

In [76]:
# Save the fits
import pandas as pd

rows = []
for fname, results in parabola_results.items():
    for fit in results['maxima']:
        rows.append({
            'file':     fname,
            'type':     'maximum',
            'mu':       fit['mu'][0],
            'mu_std':   fit['mu'][1],
            'C':        fit['C'][0],
            'C_std':    fit['C'][1],
            'A':        fit['A'][0],
            'A_std':    fit['A'][1],
        })
    for fit in results['minima']:
        rows.append({
            'file':     fname,
            'type':     'minimum',
            'mu':       fit['mu'][0],
            'mu_std':   fit['mu'][1],
            'C':        fit['C'][0],
            'C_std':    fit['C'][1],
            'A':        fit['A'][0],
            'A_std':    fit['A'][1],
        })

fits_df = pd.DataFrame(rows)
fits_df.to_csv('parabola_fits.csv', index=False)

In [77]:
# Import w/
import pandas as pd

fits_df = pd.read_csv('parabola_fits.csv')

parabola_results = {}
for fname, group in fits_df.groupby('file'):
    parabola_results[fname] = {'maxima': [], 'minima': []}
    for _, row in group.iterrows():
        fit = {
            'mu': (row['mu'],     row['mu_std']),
            'C':  (row['C'],      row['C_std']),
            'A':  (row['A'],      row['A_std']),
        }
        parabola_results[fname][row['type'] + 'a'].append(fit)

KeyError: 'maximuma'